# FDSI Dynamic Contraction Benchmark — Exploration & Evaluation

This notebook covers:
1. **Single-recording view** — RMS map, wrist angle profile, exerted force
2. **Yield** — number of decomposed MUs per condition × noise level (boxplots)
3. **Accuracy** — precision / recall / F1 vs ground truth (boxplots)

Set the paths in **Cell 2** before running.

In [ ]:
import os, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
from muniverse.evaluation.evaluate import best_time_shift, match_spikes

sns.set_theme(style='whitegrid', font_scale=1.1)
PALETTE = sns.color_palette('tab10')

## Configuration — edit here

In [ ]:
OUTPUT_DIR   = '../outputs/fdsi_benchmark'
MUSCLE       = 'FDSI'
FS           = 2048
ISO_END_S    = 5.0          # isometric window that was decomposed
TOL_S        = 0.001        # spike-matching tolerance (1 ms)
MAX_SHIFT_S  = 0.005        # max temporal shift when matching spikes

# ── Recording to visualise in Figure 1 ──────────────────────────────
VIS_SUBJECT   = 'sub-01'
VIS_CONDITION = 'triangular-ramp20s'
VIS_SNR_DB    = 20          # which noisy version to show

# ── Dataset parameters ───────────────────────────────────────────────
SUBJECT_SEEDS  = [0, 1, 2, 3, 4]
SUBJECTS       = [f'sub-{s+1:02d}' for s in SUBJECT_SEEDS]
RAMP_DURATIONS = [40, 20, 10, 5]
CONDITIONS     = [f'triangular-ramp{r}s' for r in RAMP_DURATIONS] + ['staircase']
SNR_LEVELS_DB  = [30, 25, 20, 15]

CONDITION_LABELS = {
    'triangular-ramp40s': 'Triangular 40 s',
    'triangular-ramp20s': 'Triangular 20 s',
    'triangular-ramp10s': 'Triangular 10 s',
    'triangular-ramp5s':  'Triangular 5 s',
    'staircase':          'Staircase',
}

print('Configuration OK')
print(f'  Visualising: {VIS_SUBJECT} / {VIS_CONDITION} / {VIS_SNR_DB} dB SNR')

---
## Helper functions

In [ ]:
def clean_path(subject, condition, key):
    return os.path.join(OUTPUT_DIR, 'data', subject, 'clean',
                        f'{subject}_{MUSCLE}_{condition}_{key}.npz')

def noisy_path(subject, condition, snr_db):
    return os.path.join(OUTPUT_DIR, 'data', subject, 'noisy',
                        f'{subject}_{MUSCLE}_{condition}_snr{snr_db}dB_emg.npz')

def cbss_path(subject, condition, snr_db):
    return os.path.join(OUTPUT_DIR, 'decomposition', subject,
                        f'{subject}_{MUSCLE}_{condition}_snr{snr_db}dB_cbss.npz')

def load_gt_spikes(subject, condition):
    """Return list of numpy arrays (sample indices), one per MU."""
    path = clean_path(subject, condition, 'spikes')
    return list(np.load(path, allow_pickle=True)['spikes'])

def load_cbss_results(subject, condition, snr_db):
    """Return (sources, silhouette, spikes_dict) from a CBSS .npz."""
    path = cbss_path(subject, condition, snr_db)
    if not os.path.exists(path):
        return None, None, {}
    d = np.load(path, allow_pickle=False)
    sources    = d['sources']
    silhouette = d['silhouette']
    spikes     = {
        int(k.replace('spikes_unit', '')): d[k]
        for k in d.files if k.startswith('spikes_unit')
    }
    return sources, silhouette, spikes

def compute_accuracy(gt_spikes_list, est_spikes_dict, iso_end_s, tol_s, max_shift_s, fs):
    """
    For each estimated MU find its best-matching GT MU (highest F1 within the
    isometric window) and return per-estimated-MU metrics.
    """
    iso_end_samp = int(iso_end_s * fs)

    # GT spikes filtered to iso window, converted to seconds
    gt_sec = [
        s[s < iso_end_samp] / fs
        for s in gt_spikes_list
        if len(s[s < iso_end_samp]) > 0
    ]

    rows = []
    for uid, est_idx in est_spikes_dict.items():
        if len(est_idx) == 0:
            continue
        est_sec = est_idx / fs

        best = {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
        for gt in gt_sec:
            tp, fp, fn, _ = best_time_shift(
                gt, est_sec, tolerance=tol_s,
                max_shift=max_shift_s, shift_step=tol_s / 2,
            )
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
            if f1 > best['f1']:
                best = {'precision': prec, 'recall': rec, 'f1': f1}

        rows.append(best)
    return rows

print('Helpers defined')

---
## Figure 1 — Single recording: RMS map · Wrist angle · Exerted force

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────
emg_noisy = np.load(noisy_path(VIS_SUBJECT, VIS_CONDITION, VIS_SNR_DB))['emg']  # (T, 320)
angle     = np.load(clean_path(VIS_SUBJECT, VIS_CONDITION, 'angle'))['angle']   # (T,)
effort    = np.load(clean_path(VIS_SUBJECT, VIS_CONDITION, 'effort'))['effort'] # (T,)

with open(
    os.path.join(OUTPUT_DIR, 'data', VIS_SUBJECT, 'clean',
                 f'{VIS_SUBJECT}_{MUSCLE}_{VIS_CONDITION}_metadata.json')
) as fh:
    meta = json.load(fh)

T      = emg_noisy.shape[0]
t_axis = np.arange(T) / FS   # seconds

# RMS per electrode (spatial)
rms_flat = np.sqrt(np.mean(emg_noisy ** 2, axis=0))   # (320,)
rms_grid = rms_flat.reshape(10, 32)                    # rows × cols

# ── Figure ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(3, 1, height_ratios=[1.6, 1, 1], hspace=0.45)

# ── Panel A: RMS heatmap ─────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0])
im  = ax0.imshow(rms_grid, aspect='auto', cmap='hot', origin='upper')
plt.colorbar(im, ax=ax0, label='RMS (a.u.)', fraction=0.02, pad=0.02)
ax0.set_xlabel('Electrode column')
ax0.set_ylabel('Electrode row')
ax0.set_title(
    f'A — RMS across channels   '
    f'{VIS_SUBJECT} · {CONDITION_LABELS[VIS_CONDITION]} · {VIS_SNR_DB} dB SNR'
)
ax0.set_xticks(np.arange(0, 32, 4))
ax0.set_yticks(np.arange(0, 10, 2))

# ── Panel B: Wrist angle ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[1])
ax1.plot(t_axis, angle, color='steelblue', lw=1.2)
ax1.axhline(0, color='k', lw=0.5, ls='--')
ax1.set_ylabel('Wrist angle (°)')
ax1.set_title('B — Wrist angle profile')
ax1.set_xlim(0, t_axis[-1])
ax1.set_xlabel('')
plt.setp(ax1.get_xticklabels(), visible=False)

# ── Panel C: Exerted force ───────────────────────────────────────────
ax2 = fig.add_subplot(gs[2], sharex=ax1)
ax2.plot(t_axis, effort * 100, color='firebrick', lw=1.2)
ax2.set_ylabel('Force (% MVC)')
ax2.set_xlabel('Time (s)')
ax2.set_title('C — Exerted force profile')
ax2.set_ylim(0, max(effort.max() * 110, 5))

fig.suptitle(
    f'FDSI · {CONDITION_LABELS[VIS_CONDITION]} · {VIS_SNR_DB} dB SNR',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, f'fig1_recording_{VIS_SUBJECT}_{VIS_CONDITION}_{VIS_SNR_DB}dB.pdf'),
    bbox_inches='tight'
)
plt.show()
print(f'EMG: {emg_noisy.shape}   Angle range: {angle.min():.1f} – {angle.max():.1f} °')

---
## Aggregate decomposition results across all subjects / conditions / SNR levels

In [ ]:
rows_yield  = []   # number of identified MUs
rows_acc    = []   # spike-matching accuracy per identified MU

for subject in SUBJECTS:
    for condition in CONDITIONS:
        gt_spikes = load_gt_spikes(subject, condition)
        if not gt_spikes:
            print(f'  No GT spikes: {subject} / {condition}')
            continue

        for snr_db in SNR_LEVELS_DB:
            sources, sil, est_spikes = load_cbss_results(subject, condition, snr_db)

            n_mus = len(est_spikes)
            rows_yield.append({
                'subject':   subject,
                'condition': CONDITION_LABELS[condition],
                'snr_db':    snr_db,
                'n_mus':     n_mus,
            })

            if n_mus > 0:
                acc_rows = compute_accuracy(
                    gt_spikes, est_spikes,
                    iso_end_s=ISO_END_S,
                    tol_s=TOL_S,
                    max_shift_s=MAX_SHIFT_S,
                    fs=FS,
                )
                for r in acc_rows:
                    rows_acc.append({
                        'subject':   subject,
                        'condition': CONDITION_LABELS[condition],
                        'snr_db':    snr_db,
                        **r,
                    })

df_yield = pd.DataFrame(rows_yield)
df_acc   = pd.DataFrame(rows_acc)

print(f'Yield records : {len(df_yield)}')
print(f'Accuracy records (per identified MU) : {len(df_acc)}')
print('\nYield summary:')
print(df_yield.groupby(['condition', 'snr_db'])['n_mus'].describe().round(1))

---
## Figure 2 — Decomposition yield: number of identified MUs

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

condition_order = [CONDITION_LABELS[c] for c in CONDITIONS]
snr_order       = sorted(SNR_LEVELS_DB, reverse=True)  # 30, 25, 20, 15 (left to right)

sns.boxplot(
    data      = df_yield,
    x         = 'snr_db',
    y         = 'n_mus',
    hue       = 'condition',
    order     = snr_order,
    hue_order = condition_order,
    palette   = PALETTE[:len(CONDITIONS)],
    linewidth = 0.8,
    fliersize = 3,
    ax        = ax,
)
sns.stripplot(
    data      = df_yield,
    x         = 'snr_db',
    y         = 'n_mus',
    hue       = 'condition',
    order     = snr_order,
    hue_order = condition_order,
    palette   = PALETTE[:len(CONDITIONS)],
    dodge     = True,
    alpha     = 0.55,
    size      = 4,
    jitter    = True,
    legend    = False,
    ax        = ax,
)

ax.set_xlabel('SNR (dB)')
ax.set_ylabel('Number of identified MUs')
ax.set_title(
    'CBSS decomposition yield — first 5 s isometric phase\n'
    f'FDSI · 15 % MVC · n={len(SUBJECTS)} subjects (pooled)'
)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:len(CONDITIONS)], labels[:len(CONDITIONS)],
          title='Condition', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig2_mu_yield.pdf'), bbox_inches='tight')
plt.show()

---
## Figure 3 — Accuracy metrics vs ground truth

In [ ]:
if df_acc.empty:
    print('No accuracy data available — run decomposition first.')
else:
    metrics = ['precision', 'recall', 'f1']
    metric_labels = {'precision': 'Precision', 'recall': 'Recall', 'f1': 'F1 score'}

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

    for ax, metric in zip(axes, metrics):
        sns.boxplot(
            data      = df_acc,
            x         = 'snr_db',
            y         = metric,
            hue       = 'condition',
            order     = snr_order,
            hue_order = condition_order,
            palette   = PALETTE[:len(CONDITIONS)],
            linewidth = 0.8,
            fliersize = 2,
            ax        = ax,
        )
        ax.set_xlabel('SNR (dB)')
        ax.set_ylabel(metric_labels[metric])
        ax.set_title(metric_labels[metric])
        ax.set_ylim(-0.05, 1.05)
        ax.axhline(1.0, color='k', lw=0.5, ls='--', alpha=0.4)
        ax.get_legend().remove()

    # Shared legend on last axis
    handles, labels = axes[-1].get_legend_handles_labels()
    # Re-draw legend after remove() would have cleared it:
    sns.boxplot(
        data=df_acc, x='snr_db', y='f1', hue='condition',
        order=snr_order, hue_order=condition_order,
        palette=PALETTE[:len(CONDITIONS)],
        linewidth=0.8, fliersize=2, ax=axes[-1],
    )
    handles, labels = axes[-1].get_legend_handles_labels()
    axes[-1].legend(
        handles[:len(CONDITIONS)], labels[:len(CONDITIONS)],
        title='Condition', bbox_to_anchor=(1.01, 1),
        loc='upper left', fontsize=9
    )

    fig.suptitle(
        'CBSS spike-matching accuracy vs ground truth — first 5 s isometric phase\n'
        f'FDSI · 15 % MVC · tolerance = {int(TOL_S*1000)} ms · n={len(SUBJECTS)} subjects',
        fontsize=11, fontweight='bold',
    )
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig3_accuracy.pdf'), bbox_inches='tight')
    plt.show()

    print('\nOverall accuracy summary (all conditions + SNR pooled):')
    print(df_acc[metrics].describe().round(3))

---
## Optional: Per-condition accuracy summary table

In [ ]:
if not df_acc.empty:
    summary = (
        df_acc
        .groupby(['condition', 'snr_db'])[['precision', 'recall', 'f1']]
        .agg(['mean', 'std'])
        .round(3)
    )
    display(summary)